In [0]:
# Paths
base_path = "/Volumes/workspace/default/my_volume/flight_project"

bronze_flights_path = f"{base_path}/bronze/flights"
bronze_weather_path = f"{base_path}/bronze/weather"
bronze_rejected_path = f"{base_path}/bronze/rejected"
checkpoint_flights   = f"{base_path}/checkpoints/bronze_flights"
checkpoint_weather   = f"{base_path}/checkpoints/bronze_weather"

print("Paths set!")

Paths set!


In [0]:
import requests

response = requests.get("https://opensky-network.org/api/states/all")
data = response.json()

print(f"Total flights: {len(data['states'])}")
print(f"Sample flight: {data['states'][0]}")

Total flights: 13330
Sample flight: ['39de4f', 'TVF98NQ ', 'France', 1782408399, 1782408399, 10.1897, 44.6333, 11582.4, False, 240.83, 314.31, -0.33, None, 12169.14, '1000', False, 0]


In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.types import *
from pyspark.sql.functions import *

#schema
flight_schema = StructType([
    StructField("icao24",         StringType()),
    StructField("callsign",       StringType()),
    StructField("origin_country", StringType()),
    StructField("time_position",  LongType()),
    StructField("last_contact",   LongType()),
    StructField("longitude",      DoubleType()),
    StructField("latitude",       DoubleType()),
    StructField("baro_altitude",  DoubleType()),
    StructField("on_ground",      BooleanType()),
    StructField("velocity",       DoubleType()),
    StructField("true_track",     DoubleType()),
    StructField("vertical_rate",  DoubleType()),
    StructField("sensors",        StringType()),
    StructField("geo_altitude",   DoubleType()),
    StructField("squawk",         StringType()),
    StructField("spi",            BooleanType()),
    StructField("position_source",IntegerType()),
])


# States list - DataFrame 
states = data['states']
df_flights = spark.createDataFrame(states, schema=flight_schema)

# Ingestion timestamp 
df_flights = df_flights.withColumn("ingested_at", current_timestamp())

df_flights.show(5)
print(f"Total rows: {df_flights.count()}")

+------+--------+--------------+-------------+------------+---------+--------+-------------+---------+--------+----------+-------------+-------+------------+------+-----+---------------+--------------------+
|icao24|callsign|origin_country|time_position|last_contact|longitude|latitude|baro_altitude|on_ground|velocity|true_track|vertical_rate|sensors|geo_altitude|squawk|  spi|position_source|         ingested_at|
+------+--------+--------------+-------------+------------+---------+--------+-------------+---------+--------+----------+-------------+-------+------------+------+-----+---------------+--------------------+
|39de4f|TVF98NQ |        France|   1782408399|  1782408399|  10.1897| 44.6333|      11582.4|    false|  240.83|    314.31|        -0.33|   NULL|    12169.14|  1000|false|              0|2026-06-25 17:30:...|
|a89ea5|N6545H  | United States|   1782408187|  1782408187| -98.1057| 35.7965|       624.84|    false|   57.85|       5.1|         -2.6|   NULL|       609.6|  NULL|fals

In [0]:
# Valid flights 
df_valid = df_flights.filter(
    col("icao24").isNotNull() &
    col("latitude").isNotNull() &
    col("longitude").isNotNull() &
    col("callsign").isNotNull()
)

# Rejected flights — incomplete/bad data
df_rejected = df_flights.filter(
    col("icao24").isNull() |
    col("latitude").isNull() |
    col("longitude").isNull() |
    col("callsign").isNull()
).withColumn("rejection_reason", lit("NULL values in critical fields"))

df_valid.write.format("delta") \
    .mode("overwrite") \
    .save(bronze_flights_path)

df_rejected.write.format("delta") \
    .mode("overwrite") \
    .save(bronze_rejected_path)

print(f"✅ Valid flights saved:    {df_valid.count()}")
print(f"❌ Rejected flights saved: {df_rejected.count()}")


✅ Valid flights saved:    13213
❌ Rejected flights saved: 117


In [0]:
from pyspark.sql.functions import current_timestamp
import requests
from datetime import datetime

WEATHER_API_KEY = "dc3260f4dcfc039cb0c347ce2eef3210"


airports = {
    # Europe
    "CDG": (49.0097,   2.5479),
    "LHR": (51.4775,  -0.4614),
    "FRA": (50.0379,   8.5622),
    "AMS": (52.3086,   4.7639),
    "IST": (40.9769,  28.8146),
    "MAD": (40.4719,  -3.5626),
    "FCO": (41.8003,  12.2389),
    # Middle East
    "DXB": (25.2532,  55.3657),
    "DOH": (25.2731,  51.6080),
    "AUH": (24.4330,  54.6511),
    # Asia
    "SIN": ( 1.3644, 103.9915),
    "HKG": (22.3080, 113.9185),
    "NRT": (35.7720, 140.3929),
    "BOM": (19.0896,  72.8656),
    "DEL": (28.5562,  77.1000),
    # Americas
    "JFK": (40.6413,  -73.7781),
    "LAX": (33.9425, -118.4081),
    "ORD": (41.9742,  -87.9073),
    "GRU": (-23.4356, -46.4731),
    # Pakistan
    "KHI": (24.9008,  67.1681),
    "LHE": (31.5216,  74.4037),
}

weather_data = []

for airport_code, (lat, lon) in airports.items():
    response = requests.get(
        "https://api.openweathermap.org/data/2.5/weather",
        params={
            "lat": lat,
            "lon": lon,
            "appid": WEATHER_API_KEY,
            "units": "metric"
        }
    )
    w = response.json()
    weather_data.append({
        "airport_code": airport_code,
        "temperature":  w["main"]["temp"],
        "weather_main": w["weather"][0]["main"],
        "weather_desc": w["weather"][0]["description"],
        "wind_speed":   w["wind"]["speed"],
        "visibility":   w.get("visibility", None),
        "fetched_at":   datetime.now().isoformat()
    })
    print(f"✅ {airport_code}: {w['weather'][0]['main']} — {w['main']['temp']}°C")

print(f"\nTotal airports fetched: {len(weather_data)}")

✅ CDG: Clouds — 34.68°C
✅ LHR: Clear — 35.42°C
✅ FRA: Clouds — 41.41°C
✅ AMS: Clouds — 35.59°C
✅ IST: Clear — 29.85°C
✅ MAD: Clouds — 31.49°C
✅ FCO: Clear — 32.59°C
✅ DXB: Clear — 31.8°C
✅ DOH: Clear — 35.91°C
✅ AUH: Clear — 32.4°C
✅ SIN: Clouds — 30.03°C
✅ HKG: Rain — 27.78°C
✅ NRT: Clouds — 20.95°C
✅ BOM: Clouds — 29.37°C
✅ DEL: Clouds — 39.72°C
✅ JFK: Clouds — 23.21°C
✅ LAX: Clouds — 17.74°C
✅ ORD: Clouds — 17.68°C
✅ GRU: Clouds — 19°C
✅ KHI: Clouds — 29.34°C
✅ LHE: Clear — 39.74°C

Total airports fetched: 21


In [0]:
from pyspark.sql.types import *

weather_schema = StructType([
    StructField("airport_code", StringType()),
    StructField("temperature",  DoubleType()),
    StructField("weather_main", StringType()),
    StructField("weather_desc", StringType()),
    StructField("wind_speed",   DoubleType()),
    StructField("visibility",   LongType()),
    StructField("fetched_at",   StringType()),
])

df_weather = spark.createDataFrame(weather_data, schema=weather_schema)

# Bronze mein save karo
df_weather.write.format("delta") \
    .mode("overwrite") \
    .save(bronze_weather_path)

df_weather.show()
print("✅ Weather Bronze saved!")

+------------+-----------+------------+----------------+----------+----------+--------------------+
|airport_code|temperature|weather_main|    weather_desc|wind_speed|visibility|          fetched_at|
+------------+-----------+------------+----------------+----------+----------+--------------------+
|         CDG|      34.68|      Clouds|   broken clouds|      4.36|      1640|2026-06-26T15:06:...|
|         LHR|      35.42|       Clear|       clear sky|      4.02|     10000|2026-06-26T15:06:...|
|         FRA|      41.41|      Clouds|      few clouds|      1.34|     10000|2026-06-26T15:06:...|
|         AMS|      35.59|      Clouds| overcast clouds|      3.58|     10000|2026-06-26T15:06:...|
|         IST|      29.85|       Clear|       clear sky|      6.26|     10000|2026-06-26T15:06:...|
|         MAD|      31.49|      Clouds|scattered clouds|      3.82|     10000|2026-06-26T15:06:...|
|         FCO|      32.59|       Clear|       clear sky|      4.04|     10000|2026-06-26T15:06:...|


In [0]:
print("=== BRONZE SUMMARY ===\n")

df_b_flights = spark.read.format("delta").load(bronze_flights_path)
df_b_weather = spark.read.format("delta").load(bronze_weather_path)
df_b_rejected = spark.read.format("delta").load(bronze_rejected_path)

print(f"✅ Bronze flights:  {df_b_flights.count()} rows")
print(f"✅ Bronze weather:  {df_b_weather.count()} rows")
print(f"❌ Bronze rejected: {df_b_rejected.count()} rows")

print("\n--- Sample Flights ---")
df_b_flights.select("icao24","callsign","origin_country",
                    "velocity","baro_altitude","on_ground",
                    "ingested_at").show(3)

print("\n--- Sample Weather ---")
df_b_weather.show(3)

=== BRONZE SUMMARY ===

✅ Bronze flights:  13213 rows
✅ Bronze weather:  21 rows
❌ Bronze rejected: 117 rows

--- Sample Flights ---
+------+--------+--------------------+--------+-------------+---------+--------------------+
|icao24|callsign|      origin_country|velocity|baro_altitude|on_ground|         ingested_at|
+------+--------+--------------------+--------+-------------+---------+--------------------+
|addd59|N99265  |       United States|   55.96|       853.44|    false|2026-06-25 17:32:...|
|896681|UAE387  |United Arab Emirates|  260.14|      12192.0|    false|2026-06-25 17:32:...|
|896680|UAE41M  |United Arab Emirates|  249.96|      12496.8|    false|2026-06-25 17:32:...|
+------+--------+--------------------+--------+-------------+---------+--------------------+
only showing top 3 rows

--- Sample Weather ---
+------------+-----------+------------+-------------+----------+----------+--------------------+
|airport_code|temperature|weather_main| weather_desc|wind_speed|visibil